[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/onnx/tutorials/blob/main/07_ONNX_Runtime/02_Execution_Providers/Execution_Providers_Deep_Dive.ipynb)

# Execution Providers — Deep Dive

## Table of Contents
| # | Section | Description |
|---|---------|-------------|
| 1 | [EP Concept](#1) | What EPs are and why they exist |
| 2 | [CPU EP](#2) | MLAS kernels, vectorization, threading |
| 3 | [CUDA EP](#3) | GPU execution, cuDNN, memory management |
| 4 | [TensorRT EP](#4) | Compilation-based optimization, precision |
| 5 | [OpenVINO EP](#5) | Intel hardware optimization |
| 6 | [Fallback Chain](#6) | Priority ordering and graceful degradation |
| 7 | [Kernel Dispatch](#7) | How operators map to implementations |
| 8 | [Cross-EP Transfers](#8) | Memory copies at EP boundaries |
| 9 | [Custom EPs](#9) | Building your own Execution Provider |
| 10 | [Visualization](#10) | EP comparison charts |

In [ ]:
!pip install onnxruntime onnx numpy matplotlib -q

<a id='1'></a>
## 1. The Execution Provider Concept

An **Execution Provider (EP)** is ORT's abstraction for a hardware backend. Each EP is a plugin that answers two fundamental questions:

1. **"Which operators can I execute?"** — The EP declares its capabilities by inspecting each node's op_type, input dtypes, and shapes
2. **"How do I execute them?"** — The EP provides kernel implementations or compiles subgraphs into opaque engines

### The EP Interface

Every EP implements `IExecutionProvider` with these critical methods:

```
┌─────────────────────────────────────────────────────────────────┐
│                    IExecutionProvider                            │
├─────────────────────────────────────────────────────────────────┤
│  GetCapability(GraphViewer&)                                     │
│    → vector<IndexedSubGraph>                                     │
│    Declares which nodes/subgraphs this EP can handle            │
│                                                                  │
│  Compile(vector<FusedNodeAndGraph>&)                             │
│    → Status                                                      │
│    Compiles claimed subgraphs into executable kernels            │
│                                                                  │
│  GetAllocator(OrtMemType)                                        │
│    → AllocatorPtr                                                │
│    Returns memory allocator for this EP's device                 │
│                                                                  │
│  GetDataTransfer()                                               │
│    → IDataTransfer&                                              │
│    Handles data movement between host and device                 │
└─────────────────────────────────────────────────────────────────┘
```

### EP Categories

EPs fall into two architectural categories:

| Category | Examples | Strategy |
|----------|----------|----------|
| **Kernel-based** | CPU, CUDA | Register individual kernel implementations per op |
| **Compilation-based** | TensorRT, OpenVINO | Compile entire subgraphs into fused engines |

The performance tradeoff:

$$T_{\text{kernel-based}} = \sum_{i=1}^{n} T_{\text{kernel}_i} + (n-1) \cdot T_{\text{dispatch}}$$

$$T_{\text{compilation-based}} = T_{\text{compiled\_engine}} + T_{\text{compile}} / N_{\text{runs}}$$

For large $N_{\text{runs}}$, compilation-based EPs amortize their upfront cost and win on steady-state throughput.

In [ ]:
import onnxruntime as ort

print("ONNX Runtime Execution Providers")
print("=" * 50)
providers = ort.get_available_providers()
for i, ep in enumerate(providers):
    print(f"  [{i}] {ep}")

print(f"\nTotal available: {len(providers)}")
print(f"\nNote: Priority order matters — first EP gets first refusal on nodes.")
print(f"CPU EP is always available as the universal fallback.")

<a id='2'></a>
## 2. CPU Execution Provider — MLAS and Vectorization

The CPU EP is ORT's default and always-available backend. It leverages **MLAS** (Microsoft Linear Algebra Subroutines) for compute-intensive operations.

### MLAS Architecture

```
┌────────────────────────────────────────────────────────────┐
│                     MLAS Library                            │
├────────────────────────────────────────────────────────────┤
│  GEMM Engine                                               │
│    ├── AVX-512 path (Intel Xeon, 16 FP32 per cycle)       │
│    ├── AVX2 path (mainstream x86, 8 FP32 per cycle)       │
│    ├── NEON path (ARM, 4 FP32 per cycle)                  │
│    └── Scalar fallback                                     │
├────────────────────────────────────────────────────────────┤
│  Convolution Engine                                        │
│    ├── im2col + GEMM (general case)                       │
│    ├── Direct convolution (small kernels)                  │
│    └── Winograd (3x3 kernels, stride 1)                   │
├────────────────────────────────────────────────────────────┤
│  Quantized Kernels                                         │
│    ├── INT8 GEMM (VNNI on Ice Lake+)                      │
│    ├── INT8 Conv                                           │
│    └── Mixed-precision accumulation                        │
└────────────────────────────────────────────────────────────┘
```

### GEMM Performance Model

For matrix multiply $C = A \times B$ with $A \in \mathbb{R}^{M \times K}$, $B \in \mathbb{R}^{K \times N}$:

$$\text{FLOPs} = 2MKN$$

$$\text{Peak throughput}_{\text{AVX-512}} = \text{clock} \times 2 \times 16 = 2 \times 3\text{GHz} \times 32 = 192 \text{ GFLOPS (single core)}$$

The **arithmetic intensity** determines whether the kernel is compute or memory bound:

$$\text{AI} = \frac{2MKN}{4(MK + KN + MN)} \quad [\text{FLOP/byte}]$$

For square matrices ($M = K = N$):

$$\text{AI} = \frac{2N^3}{4 \cdot 3N^2} = \frac{N}{6}$$

So for $N > 6 \times \text{ops:byte ratio}$, the GEMM becomes compute-bound. On modern CPUs with ~10 FLOP/byte bandwidth ratio, this happens at $N \approx 60$.

In [ ]:
import numpy as np
import onnx
from onnx import helper, TensorProto, numpy_helper
import onnxruntime as ort
import time

# Build models of different sizes to observe MLAS performance
def build_matmul_model(M, K, N):
    W = np.random.randn(K, N).astype(np.float32)
    X = helper.make_tensor_value_info("X", TensorProto.FLOAT, [M, K])
    Y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, [M, N])
    graph = helper.make_graph(
        [helper.make_node("MatMul", ["X", "W"], ["Y"])],
        "gemm_bench", [X], [Y],
        initializer=[numpy_helper.from_array(W, "W")]
    )
    model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 17)])
    return model

# Benchmark different matrix sizes
sizes = [(32, 32, 32), (64, 64, 64), (128, 128, 128), (256, 256, 256), 
         (512, 512, 512), (1024, 1024, 1024)]

print(f"{'M×K×N':>15} {'GFLOPS':>10} {'Time(ms)':>10} {'AI(FLOP/B)':>12} {'Bound':>10}")
print("-" * 60)

gflops_results = []
for M, K, N in sizes:
    model = build_matmul_model(M, K, N)
    onnx.save(model, "_bench.onnx")
    
    sess = ort.InferenceSession("_bench.onnx", providers=["CPUExecutionProvider"])
    x = np.random.randn(M, K).astype(np.float32)
    
    # Warmup
    for _ in range(50):
        sess.run(None, {"X": x})
    
    # Measure
    start = time.perf_counter()
    n_iter = 200
    for _ in range(n_iter):
        sess.run(None, {"X": x})
    elapsed = (time.perf_counter() - start) / n_iter * 1000  # ms
    
    flops = 2 * M * K * N
    gflops = flops / (elapsed / 1000) / 1e9
    ai = flops / (4 * (M*K + K*N + M*N))
    bound = "Compute" if ai > 10 else "Memory"
    
    gflops_results.append((M, gflops, ai))
    print(f"{M}×{K}×{N:>4} {gflops:>10.2f} {elapsed:>10.4f} {ai:>12.1f} {bound:>10}")

import os
os.remove("_bench.onnx")

<a id='3'></a>
## 3. CUDA Execution Provider

The CUDA EP targets NVIDIA GPUs via cuDNN (convolutions) and cuBLAS (linear algebra). Key characteristics:

### Memory Model

```
┌──────────── Host (CPU) ────────────┐     ┌──────────── Device (GPU) ────────────┐
│                                     │     │                                       │
│  ┌─────────────────┐               │     │  ┌─────────────────┐                 │
│  │  Input Tensors   │──── PCIe ────────────►│  Device Memory    │                 │
│  └─────────────────┘               │     │  │  (HBM2/GDDR6)   │                 │
│                                     │     │  └────────┬────────┘                 │
│  ┌─────────────────┐               │     │           │                           │
│  │  Output Tensors  │◄─── PCIe ────────────┤  ┌───────▼────────┐                 │
│  └─────────────────┘               │     │  │  CUDA Kernels    │                 │
│                                     │     │  │  (cuDNN, cuBLAS) │                 │
└─────────────────────────────────────┘     │  └─────────────────┘                 │
                                            └───────────────────────────────────────┘
```

### H2D/D2H Transfer Cost

The GPU execution model introduces transfer overhead. For a tensor of size $S$ bytes:

$$T_{\text{H2D}} = T_{\text{launch}} + \frac{S}{\text{BW}_{\text{PCIe}}}$$

With PCIe Gen4 x16 at 25 GB/s effective bandwidth:

$$T_{\text{H2D}}(\text{1MB}) = 5\mu s + \frac{10^6}{25 \times 10^9} \approx 45 \mu s$$

### When GPU EP Wins

GPU inference is beneficial when:

$$T_{\text{GPU\_compute}} + 2T_{\text{transfer}} < T_{\text{CPU\_compute}}$$

$$\frac{\text{FLOPs}}{\text{GPU\_TFLOPS}} + \frac{2S}{\text{BW}_{\text{PCIe}}} < \frac{\text{FLOPs}}{\text{CPU\_GFLOPS}}$$

For a model with $F$ FLOPs and input size $S$, the **crossover point** where GPU becomes faster:

$$F > \frac{2S \cdot \text{GPU\_TFLOPS}}{\text{BW}_{\text{PCIe}}} \cdot \frac{1}{1 - \text{GPU\_TFLOPS}/\text{CPU\_GFLOPS}}$$

<a id='4'></a>
## 4. TensorRT Execution Provider

TensorRT is NVIDIA's **compilation-based** inference optimizer. Unlike the CUDA EP (which dispatches individual kernels), TensorRT:

1. **Accepts a subgraph** from ORT's partitioner
2. **Compiles it** into a highly fused engine ("TRT engine")
3. **Caches the engine** for subsequent sessions

### Compilation Pipeline

```
ONNX Subgraph ──► TRT Network ──► Builder ──► Optimized Engine
     │                 │              │              │
     │                 │              │              └── Fused CUDA kernels
     │                 │              └── Layer fusion, precision calibration
     │                 └── TRT layer representation
     └── ORT claimed nodes
```

### TRT Optimizations

| Optimization | Effect | Typical Speedup |
|-------------|--------|----------------|
| Layer fusion | Reduces kernel launches and memory traffic | 1.5-3x |
| Precision reduction (FP16/INT8) | More ops per cycle, less memory | 2-4x |
| Kernel auto-tuning | Selects fastest kernel variant per layer | 1.1-1.3x |
| Memory optimization | Reuses buffers across layers | Reduces peak memory |

### FP16 Throughput Advantage

For Tensor Core-equipped GPUs (Volta+), FP16 inference provides:

$$\text{Speedup}_{\text{FP16}} = \frac{\text{TC\_TFLOPS}_{\text{FP16}}}{\text{CUDA\_TFLOPS}_{\text{FP32}}}$$

On A100: $\frac{312 \text{ TFLOPS}}{19.5 \text{ TFLOPS}} = 16\times$ theoretical peak improvement.

### Dynamic Shapes Challenge

TRT engines are compiled for specific input shapes (or shape ranges). Dynamic shapes require either:
- **Optimization profiles**: Pre-defined min/opt/max shapes
- **Multiple engines**: One per common batch size
- **Fallback to CUDA EP**: For truly dynamic dimensions

The compilation cost:

$$T_{\text{compile}} = O(|\text{layers}| \times |\text{tactics}| \times T_{\text{benchmark}})$$

This can be minutes for large models, hence engine caching is essential.

<a id='5'></a>
## 5. OpenVINO Execution Provider

OpenVINO (Open Visual Inference and Neural Network Optimization) is Intel's inference toolkit targeting Intel CPUs, integrated GPUs, VPUs, and FPGAs.

### Architecture

```
ONNX Model ──► ORT OpenVINO EP ──► OpenVINO Plugin ──► Intel Hardware
                      │                    │                   │
                      │                    ├── CPU plugin ────► Xeon/Core (AVX-512, VNNI)
                      │                    ├── GPU plugin ────► Intel UHD/Iris (Gen9+)
                      │                    ├── VPU plugin ────► Movidius Myriad
                      │                    └── FPGA plugin ──► Intel Arria/Stratix
                      │
                      └── Subgraph compiled to OpenVINO IR
```

### Key Optimizations

- **Graph-level**: Layer fusion (Conv+BN+Relu), constant folding, dead code elimination
- **Precision**: INT8 quantization with accuracy-preserving calibration
- **Memory**: In-place tensor operations, memory-optimal execution order
- **Hardware-specific**: VNNI instructions for INT8, AVX-512 for FP32

### INT8 Throughput on Intel VNNI

With VNNI (Vector Neural Network Instructions), Intel CPUs can process 4 INT8 multiply-accumulates per cycle per FP32 lane:

$$\text{INT8\_Throughput}_{\text{VNNI}} = 4 \times \text{FP32\_Throughput}$$

Combined with reduced memory bandwidth (1 byte vs 4 bytes per element):

$$\text{Effective speedup} \approx 4\times \text{(compute)} \times \frac{4}{1} \text{(bandwidth)} = 4\text{-}8\times$$

<a id='6'></a>
## 6. Fallback Chain — Priority-Based Node Assignment

When multiple EPs are registered, ORT uses a strict priority ordering for node assignment. The fallback chain ensures every node gets executed, even if the preferred EP can't handle it.

### Assignment Algorithm

```
Given: EP list [EP_0 (highest priority), EP_1, ..., EP_k (CPU, lowest)]
       Graph G = (V, E) with nodes V

for each node v in topological_sort(V):
    for j = 0 to k:
        if EP_j.CanExecute(v.op_type, v.dtypes, v.shapes):
            assignment[v] = EP_j
            break  # first capable EP wins
    # CPU EP (EP_k) is guaranteed to handle all standard ops
```

### Why Nodes Fall Through

A node may not be claimed by a preferred EP for several reasons:

| Reason | Example |
|--------|--------|
| Unsupported op type | Custom op not in TRT's registry |
| Unsupported dtype | BFloat16 on older CUDA EP |
| Shape constraints | Dynamic rank not supported by compiled engine |
| Attribute mismatch | Specific padding mode not implemented |
| Version gap | Op version newer than EP's implementation |

### Fallback Cost

Each fallback boundary introduces a device transfer:

$$T_{\text{fallback}} = T_{\text{D2H}} + T_{\text{CPU\_kernel}} + T_{\text{H2D}}$$

For a single Reshape op falling back to CPU with a 10MB tensor on PCIe Gen4:

$$T_{\text{fallback}} \approx \frac{10 \times 10^6}{25 \times 10^9} \times 2 + T_{\text{reshape}} \approx 0.8\text{ms} + \epsilon$$

This 0.8ms penalty may exceed the actual compute time of the op, making fallback expensive for small ops.

In [ ]:
import onnxruntime as ort
import onnx
from onnx import helper, TensorProto, numpy_helper
import numpy as np

# Build a model with ops that might trigger fallback
# Some ops (e.g., custom, exotic dtypes) won't be on all EPs
np.random.seed(42)
W = np.random.randn(64, 32).astype(np.float32)

X = helper.make_tensor_value_info("X", TensorProto.FLOAT, ["batch", 64])
Y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, ["batch", 32])

nodes = [
    helper.make_node("MatMul", ["X", "W"], ["mm"]),
    helper.make_node("Relu", ["mm"], ["r"]),
    helper.make_node("Sigmoid", ["r"], ["s"]),
    helper.make_node("Tanh", ["s"], ["Y"]),
]

graph = helper.make_graph(nodes, "fallback_demo", [X], [Y],
    initializer=[numpy_helper.from_array(W, "W")])
model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 17)])
onnx.save(model, "fallback_demo.onnx")

# Test with different EP configurations
available = ort.get_available_providers()
print("Testing EP configurations:")
print("=" * 50)

configs = [
    ["CPUExecutionProvider"],
]
if "CUDAExecutionProvider" in available:
    configs.append(["CUDAExecutionProvider", "CPUExecutionProvider"])
if "TensorrtExecutionProvider" in available:
    configs.append(["TensorrtExecutionProvider", "CUDAExecutionProvider", "CPUExecutionProvider"])

x_test = np.random.randn(8, 64).astype(np.float32)

for providers in configs:
    try:
        sess = ort.InferenceSession("fallback_demo.onnx", providers=providers)
        result = sess.run(None, {"X": x_test})
        print(f"  Providers: {providers}")
        print(f"  Active: {sess.get_providers()}")
        print(f"  Output shape: {result[0].shape}")
        print()
    except Exception as e:
        print(f"  Providers: {providers} -> FAILED: {e}")
        print()

<a id='7'></a>
## 7. Kernel Dispatch — From Op Type to Implementation

When ORT executes a node, it must resolve the abstract ONNX op to a concrete kernel. This dispatch process considers:

1. **Op type and version** — Different opset versions may have different semantics
2. **Input data types** — float32, float16, int8, etc.
3. **EP assignment** — Which EP owns this node
4. **Fused patterns** — Multi-node patterns that map to single kernels

### Dispatch Resolution

```
Node: op_type="Conv", version=11, inputs=[float32 NCHW tensor, float32 weight]
  │
  ├─ EP = CUDA?
  │   └─ Lookup: CUDA KernelRegistry["Conv", v11, float32]
  │       └─ Found: CudnnConvKernel (delegates to cuDNN)
  │
  ├─ EP = CPU?
  │   └─ Lookup: CPU KernelRegistry["Conv", v11, float32]
  │       └─ Found: MlasConvKernel (im2col + MLAS GEMM)
  │
  └─ EP = TensorRT?
      └─ Already compiled into TRT engine (no per-op dispatch)
```

### Fused Kernel Patterns (CPU EP)

ORT's CPU EP recognizes and fuses common patterns:

| Pattern | Fused Kernel | Benefit |
|---------|-------------|--------|
| MatMul + Add | FusedMatMul | Single GEMM call with bias |
| Conv + BN + Relu | FusedConv | BN folded into weights, Relu in-place |
| LayerNorm components | FusedLayerNorm | Single pass over data |
| Attention (Q/K/V) | FusedAttention | Flash attention pattern |
| Gelu (approximate) | FusedGelu | Avoids expensive erf |

Memory traffic reduction from fusion:

$$\text{Savings} = (k - 1) \times 2 \times |\text{tensor}| \quad \text{for k-op fusion}$$

For a MatMul+Add+Relu fusion on a $(B, 1024)$ output:

$$\text{Savings} = 2 \times 2 \times B \times 1024 \times 4 = 16B \text{ KB}$$

In [ ]:
import onnxruntime as ort
import onnx
from onnx import helper, TensorProto, numpy_helper
import numpy as np
import time

# Build a model with fusible patterns: MatMul + Add + Relu
np.random.seed(42)
W1 = np.random.randn(512, 256).astype(np.float32) * 0.01
B1 = np.zeros(256, dtype=np.float32)
W2 = np.random.randn(256, 128).astype(np.float32) * 0.01
B2 = np.zeros(128, dtype=np.float32)

X = helper.make_tensor_value_info("X", TensorProto.FLOAT, ["batch", 512])
Y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, ["batch", 128])

nodes = [
    helper.make_node("MatMul", ["X", "W1"], ["h1"]),
    helper.make_node("Add", ["h1", "B1"], ["h1b"]),
    helper.make_node("Relu", ["h1b"], ["h1r"]),
    helper.make_node("MatMul", ["h1r", "W2"], ["h2"]),
    helper.make_node("Add", ["h2", "B2"], ["h2b"]),
    helper.make_node("Relu", ["h2b"], ["Y"]),
]

graph = helper.make_graph(nodes, "fusion_demo", [X], [Y],
    initializer=[
        numpy_helper.from_array(W1, "W1"), numpy_helper.from_array(B1, "B1"),
        numpy_helper.from_array(W2, "W2"), numpy_helper.from_array(B2, "B2"),
    ])
model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 17)])
onnx.save(model, "fusion_demo.onnx")

# Compare with/without optimization (fusion)
x_test = np.random.randn(64, 512).astype(np.float32)

so_no_opt = ort.SessionOptions()
so_no_opt.graph_optimization_level = ort.GraphOptimizationLevel.ORT_DISABLE_ALL
sess_no_opt = ort.InferenceSession("fusion_demo.onnx", so_no_opt, providers=["CPUExecutionProvider"])

so_opt = ort.SessionOptions()
so_opt.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
sess_opt = ort.InferenceSession("fusion_demo.onnx", so_opt, providers=["CPUExecutionProvider"])

# Warmup
for _ in range(100):
    sess_no_opt.run(None, {"X": x_test})
    sess_opt.run(None, {"X": x_test})

# Benchmark
n_iter = 500
start = time.perf_counter()
for _ in range(n_iter):
    sess_no_opt.run(None, {"X": x_test})
t_no_opt = (time.perf_counter() - start) / n_iter * 1000

start = time.perf_counter()
for _ in range(n_iter):
    sess_opt.run(None, {"X": x_test})
t_opt = (time.perf_counter() - start) / n_iter * 1000

print(f"Kernel Fusion Impact (batch=64, 2-layer MLP):")
print(f"  Without fusion: {t_no_opt:.3f} ms/iter")
print(f"  With fusion:    {t_opt:.3f} ms/iter")
print(f"  Speedup: {t_no_opt/t_opt:.2f}x")

<a id='8'></a>
## 8. Cross-EP Transfers

When adjacent nodes are assigned to different EPs (e.g., GPU → CPU → GPU), ORT automatically inserts `MemcpyFromHost` and `MemcpyToHost` nodes to handle data transfer.

### Transfer Anatomy

```
GPU Node A ──► [MemcpyToHost] ──► CPU Node B ──► [MemcpyFromHost] ──► GPU Node C
     │              │                   │                │                  │
     │         D2H copy             CPU kernel       H2D copy              │
     │         (async)                                (async)               │
     └──────────────────────────────────────────────────────────────────────┘
                              Total overhead: 2 × PCIe transfers
```

### Minimizing Transfers

Strategies to reduce cross-EP overhead:

1. **Increase EP coverage**: Use EPs that support more ops
2. **Graph restructuring**: Move problematic ops before export
3. **IOBinding**: Keep inputs/outputs on GPU memory to avoid H2D/D2H for I/O
4. **Custom ops**: Implement missing ops on the target EP

### Cost Model

Total execution time with $k$ EP boundary crossings:

$$T_{\text{total}} = \sum_{i} T_{\text{kernel}_i} + k \cdot \left(T_{\text{sync}} + \frac{|\text{tensor}|}{\text{BW}}\right)$$

Where:
- $T_{\text{sync}} \approx 5\text{-}20 \mu s$ (CUDA stream synchronization)
- $\text{BW} \approx 12\text{-}25$ GB/s (PCIe Gen3/Gen4)
- $|\text{tensor}|$ = tensor size in bytes

<a id='9'></a>
## 9. Custom Execution Providers

ORT allows building custom EPs for proprietary hardware or specialized accelerators. The implementation requires:

### Minimal Custom EP Structure

```cpp
class MyCustomEP : public IExecutionProvider {
public:
    // Declare supported operations
    vector<unique_ptr<ComputeCapability>>
    GetCapability(const GraphViewer& graph,
                  const IKernelLookup& kernel_lookup) override {
        vector<unique_ptr<ComputeCapability>> caps;
        for (auto& node : graph.Nodes()) {
            if (CanHandle(node)) {
                caps.push_back(MakeCapability(node));
            }
        }
        return caps;
    }
    
    // Compile claimed subgraphs
    Status Compile(const vector<FusedNodeAndGraph>& fused_nodes,
                   vector<NodeComputeInfo>& node_compute_funcs) override {
        for (auto& fused : fused_nodes) {
            node_compute_funcs.push_back(CreateComputeFunc(fused));
        }
        return Status::OK();
    }
};
```

### When to Build a Custom EP

- Custom AI accelerator hardware (TPU-like, FPGA)
- Proprietary kernel library integration
- Research prototyping of new execution strategies
- Edge devices with non-standard instruction sets

<a id='10'></a>
## 10. EP Comparison Visualization

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 2, figsize=(15, 11))

# Plot 1: EP comparison radar/bar chart
eps = ['CPU\n(MLAS)', 'CUDA\n(cuDNN)', 'TensorRT', 'OpenVINO']
# Relative scores (1-10)
metrics = {
    'Throughput': [3, 7, 10, 5],
    'Latency': [4, 7, 9, 6],
    'Flexibility': [10, 8, 5, 7],
    'Setup Ease': [10, 6, 4, 5],
    'Op Coverage': [10, 9, 6, 7],
}

x = np.arange(len(eps))
width = 0.15
colors = ['#3498db', '#2ecc71', '#e74c3c', '#9b59b6', '#f39c12']

for i, (metric, scores) in enumerate(metrics.items()):
    axes[0,0].bar(x + i*width, scores, width, label=metric, color=colors[i], edgecolor='black', linewidth=0.5)

axes[0,0].set_xlabel('Execution Provider', fontsize=11)
axes[0,0].set_ylabel('Score (1-10)', fontsize=11)
axes[0,0].set_title('EP Comparison Across Metrics', fontsize=12, fontweight='bold')
axes[0,0].set_xticks(x + width * 2)
axes[0,0].set_xticklabels(eps)
axes[0,0].legend(fontsize=8, ncol=3)
axes[0,0].grid(True, alpha=0.3, axis='y')

# Plot 2: Latency breakdown by EP
categories = ['Kernel\nExec', 'Memory\nAlloc', 'Dispatch\nOverhead', 'Data\nTransfer']
cpu_breakdown = [60, 15, 20, 5]
cuda_breakdown = [30, 10, 5, 55]
trt_breakdown = [20, 5, 2, 73]

x2 = np.arange(len(categories))
width2 = 0.25

axes[0,1].bar(x2 - width2, cpu_breakdown, width2, label='CPU', color='#3498db')
axes[0,1].bar(x2, cuda_breakdown, width2, label='CUDA', color='#2ecc71')
axes[0,1].bar(x2 + width2, trt_breakdown, width2, label='TensorRT', color='#e74c3c')

axes[0,1].set_xlabel('Latency Component', fontsize=11)
axes[0,1].set_ylabel('% of Total Latency', fontsize=11)
axes[0,1].set_title('Latency Breakdown by EP', fontsize=12, fontweight='bold')
axes[0,1].set_xticks(x2)
axes[0,1].set_xticklabels(categories)
axes[0,1].legend()
axes[0,1].grid(True, alpha=0.3, axis='y')

# Plot 3: Throughput scaling with model complexity
model_sizes = ['Small\n(1M)', 'Medium\n(10M)', 'Large\n(100M)', 'XLarge\n(1B)']
cpu_tput = [5000, 800, 50, 5]
cuda_tput = [3000, 5000, 2000, 200]
trt_tput = [4000, 8000, 4000, 500]

x3 = np.arange(len(model_sizes))
axes[1,0].semilogy(x3, cpu_tput, 'bs-', linewidth=2, markersize=10, label='CPU')
axes[1,0].semilogy(x3, cuda_tput, 'go-', linewidth=2, markersize=10, label='CUDA')
axes[1,0].semilogy(x3, trt_tput, 'r^-', linewidth=2, markersize=10, label='TensorRT')

axes[1,0].set_xlabel('Model Size (Parameters)', fontsize=11)
axes[1,0].set_ylabel('Throughput (samples/sec)', fontsize=11)
axes[1,0].set_title('Throughput vs Model Complexity', fontsize=12, fontweight='bold')
axes[1,0].set_xticks(x3)
axes[1,0].set_xticklabels(model_sizes)
axes[1,0].legend(fontsize=10)
axes[1,0].grid(True, alpha=0.3)

# Plot 4: Fallback chain diagram
axes[1,1].axis('off')
axes[1,1].set_title('EP Fallback Chain', fontsize=12, fontweight='bold')

# Draw boxes and arrows
box_props = dict(boxstyle='round,pad=0.3', facecolor='lightblue', edgecolor='black')
fall_props = dict(boxstyle='round,pad=0.3', facecolor='lightyellow', edgecolor='orange')
cpu_props = dict(boxstyle='round,pad=0.3', facecolor='lightgreen', edgecolor='green')

axes[1,1].text(0.15, 0.85, 'TensorRT EP\n(Compilation)', ha='center', va='center',
              fontsize=10, bbox=box_props, transform=axes[1,1].transAxes)
axes[1,1].text(0.50, 0.85, 'CUDA EP\n(cuDNN/cuBLAS)', ha='center', va='center',
              fontsize=10, bbox=box_props, transform=axes[1,1].transAxes)
axes[1,1].text(0.85, 0.85, 'CPU EP\n(MLAS/Eigen)', ha='center', va='center',
              fontsize=10, bbox=cpu_props, transform=axes[1,1].transAxes)

# Arrows
axes[1,1].annotate('', xy=(0.35, 0.85), xytext=(0.28, 0.85),
                  arrowprops=dict(arrowstyle='->', color='red', lw=2),
                  transform=axes[1,1].transAxes)
axes[1,1].annotate('', xy=(0.70, 0.85), xytext=(0.63, 0.85),
                  arrowprops=dict(arrowstyle='->', color='red', lw=2),
                  transform=axes[1,1].transAxes)

axes[1,1].text(0.31, 0.92, 'unsupported\nops', ha='center', fontsize=7, color='red',
              transform=axes[1,1].transAxes)
axes[1,1].text(0.66, 0.92, 'no GPU /\nunsupported', ha='center', fontsize=7, color='red',
              transform=axes[1,1].transAxes)

# Example assignment
axes[1,1].text(0.5, 0.55, 'Example: ResNet-50 with TRT+CUDA+CPU', ha='center',
              fontsize=11, fontweight='bold', transform=axes[1,1].transAxes)

assignments = [
    ('Conv layers (85%)', '#2ecc71', 'TensorRT'),
    ('Custom ops (10%)', '#3498db', 'CUDA'),
    ('Reshape/Unsupported (5%)', '#f39c12', 'CPU'),
]
for i, (desc, color, ep) in enumerate(assignments):
    axes[1,1].text(0.5, 0.40 - i*0.12, f'{desc} → {ep}', ha='center',
                  fontsize=10, color=color, fontweight='bold',
                  transform=axes[1,1].transAxes)

plt.tight_layout()
plt.savefig('ep_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# GEMM performance (roofline model)
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

# Roofline parameters (illustrative CPU)
peak_flops = 200  # GFLOPS (single core AVX-512)
bandwidth = 50  # GB/s
ridge_point = peak_flops / bandwidth  # FLOP/byte

ai = np.logspace(-1, 3, 100)  # Arithmetic intensity
roofline = np.minimum(peak_flops, bandwidth * ai)

ax.loglog(ai, roofline, 'b-', linewidth=3, label='Roofline')
ax.axvline(x=ridge_point, color='gray', linestyle='--', alpha=0.5, label=f'Ridge point ({ridge_point:.1f} FLOP/B)')

# Mark GEMM sizes from our benchmark
for M, gflops, ai_val in gflops_results:
    marker = 'o' if ai_val < ridge_point else 's'
    color = 'red' if ai_val < ridge_point else 'green'
    ax.loglog(ai_val, gflops, marker, markersize=12, color=color, 
             label=f'{M}×{M} (AI={ai_val:.1f})')

ax.fill_between(ai[ai < ridge_point], 0.1, bandwidth * ai[ai < ridge_point], 
               alpha=0.1, color='red', label='Memory-bound region')
ax.fill_between(ai[ai >= ridge_point], 0.1, peak_flops, 
               alpha=0.1, color='green', label='Compute-bound region')

ax.set_xlabel('Arithmetic Intensity (FLOP/byte)', fontsize=12)
ax.set_ylabel('Performance (GFLOPS)', fontsize=12)
ax.set_title('CPU Roofline Model with GEMM Benchmarks\n(MLAS Kernel Performance)', fontsize=13, fontweight='bold')
ax.legend(fontsize=9, loc='lower right')
ax.grid(True, alpha=0.3, which='both')
ax.set_xlim([0.1, 1000])
ax.set_ylim([0.1, 500])

plt.tight_layout()
plt.savefig('ep_roofline.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cleanup
import os
for f in ['fallback_demo.onnx', 'fusion_demo.onnx']:
    if os.path.exists(f):
        os.remove(f)
print("Cleanup complete.")

## Summary

Execution Providers are ORT's pluggable hardware abstraction layer:

- **CPU EP** uses MLAS for vectorized GEMM/Conv with AVX-512/NEON intrinsics, following the roofline performance model
- **CUDA EP** delegates to cuDNN/cuBLAS for GPU kernels with H2D/D2H transfer overhead
- **TensorRT EP** compiles subgraphs into fused engines for maximum GPU throughput (16x with FP16 Tensor Cores)
- **OpenVINO EP** optimizes for Intel hardware with INT8 VNNI acceleration
- **Fallback chain** ensures graceful degradation: TRT → CUDA → CPU
- **Kernel dispatch** resolves (op_type, version, dtype) → concrete implementation
- **Cross-EP transfers** add latency proportional to tensor size / PCIe bandwidth